# 04 - Bitstamp Connector Exploration

**Goal**: Explore the Bitstamp public REST API to inform our production connector.

**Scope**:
- Test raw `httpx` approach (no existing SDK — custom connector per DEC-011)
- Map Bitstamp symbols to canonical pairs
- Parse responses into our `TopOfBook` dataclass
- Document rate limits, error handling, edge cases
- Verify which of our 8 target pairs are available
- Answer key design questions for the production connector

**Target Pairs** (from PROJECT_INSTRUCTIONS.md):
- BTC/USD, BTC/USDC
- LTC/USD, LTC/USDC, LTC/BTC
- SOL/USD, SOL/USDC, SOL/BTC

**Key Differences from Kraken, Coinbase & Gemini**:
- Bitstamp uses lowercase, no-separator symbols: `btcusd` (same as Gemini)
- No existing Python SDK for async use — raw httpx from the start
- Public endpoints: 400 req/sec, 10K per 10 minutes (very generous)
- Ticker endpoint has bid/ask prices but **NO bid/ask sizes** (same as Gemini)
- Must use order book endpoint (`/api/v2/order_book/{symbol}/`) for TopOfBook with sizes
- **Critical**: Order book entries are **arrays-of-arrays** `[[price, amount]]`, NOT arrays-of-objects
- Timestamps: `timestamp` (Unix seconds string) + `microtimestamp` (Unix microseconds string)
- Some target pairs may not be available (research suggests 5/8)
- Acquired by Robinhood (June 2025); Ohio-eligible (confirmed Phase A research)

**Lessons Applied** (from LESSONS_LEARNED.md):
- LL-001: Verify exact symbol format, don't assume
- LL-002: Document actual response shapes from live API, not just docs
- LL-003: Test rate limit behavior before building production connector
- LL-010: All prices/sizes via `to_decimal()`, never float
- LL-050: Use `nest_asyncio.apply()` for async in Jupyter
- LL-052: No batch endpoint assumption — verify before building connector
- LL-060: Ticker endpoints often lack bid/ask sizes — verify and use order book if needed

## 1. Setup

In [1]:
# Install dependencies (run once)
# !pip install httpx nest_asyncio

In [2]:
import asyncio
import json
import sys
import time
from decimal import Decimal
from pprint import pprint

import httpx
import nest_asyncio

# Enable nested event loops for Jupyter (LL-050)
nest_asyncio.apply()

sys.path.insert(0, "../src")

# Our existing infrastructure — reuse, don't reimplement
from uscryptoarb.marketdata.topofbook import TopOfBook, tob_from_raw
from uscryptoarb.misc.decimals import to_decimal
from uscryptoarb.validation.guards import require_present
from uscryptoarb.venues.symbol_translator import SymbolTranslator, create_translator

BASE_URL = "https://www.bitstamp.net"

print("Setup complete.")

Setup complete.


## 2. Symbol Discovery & Mapping

Bitstamp uses lowercase, no-separator symbols: `btcusd`, `ltcbtc`, `solusd`.
Same format as Gemini. Let's fetch all available markets and check which of our
8 target pairs exist.

**Key question**: Research suggests only 5/8 target pairs are available.
Missing: LTC/USDC, SOL/USDC, SOL/BTC. Verify here.

In [3]:
# Fetch all available markets via /api/v2/markets/
# (replaces deprecated /api/v2/trading-pairs-info/)
resp = httpx.get(f"{BASE_URL}/api/v2/markets/")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}")

markets_data = resp.json()
print(f"\nResponse type: {type(markets_data)}")

# Inspect structure — may be a list or dict
if isinstance(markets_data, list):
    print(f"Total markets: {len(markets_data)}")
    if markets_data:
        print(f"\nFirst entry structure:")
        pprint(markets_data[0])
elif isinstance(markets_data, dict):
    print(f"Top-level keys: {list(markets_data.keys())}")
    # Might be nested under a key
    for key, val in markets_data.items():
        if isinstance(val, list):
            print(f"\n{key}: {len(val)} items")
            if val:
                print(f"First entry:")
                pprint(val[0])

Status: 200
Content-Type: application/json

Response type: <class 'list'>
Total markets: 295

First entry structure:
{'base_currency': 'BTC',
 'base_decimals': 8,
 'counter_currency': 'USD',
 'counter_decimals': 0,
 'description': 'Bitcoin / U.S. dollar',
 'instant_and_market_orders': 'Enabled',
 'instant_order_counter_decimals': 2,
 'market_symbol': 'btcusd',
 'market_type': 'SPOT',
 'minimum_order_value': '10',
 'name': 'BTC/USD',
 'trading': 'Enabled'}


In [4]:
# Extract all available market symbols
# Adapt field name based on actual response structure above
# Expected fields: 'url_symbol', 'name', or 'pair' depending on API version

all_symbols = set()

if isinstance(markets_data, list):
    for m in markets_data:
        # Try common field names
        sym = m.get("url_symbol") or m.get("pair") or m.get("name", "")
        if sym:
            all_symbols.add(sym.lower())
elif isinstance(markets_data, dict):
    # Might need to iterate nested structure
    for key, val in markets_data.items():
        if isinstance(val, list):
            for m in val:
                sym = m.get("url_symbol") or m.get("pair") or m.get("name", "")
                if sym:
                    all_symbols.add(sym.lower())

print(f"Total unique symbols extracted: {len(all_symbols)}")
print(f"\nSample symbols (sorted): {sorted(all_symbols)[:20]}")

Total unique symbols extracted: 295

Sample symbols (sorted): ['1inch/eur', '1inch/usd', 'aave/btc', 'aave/eur', 'aave/usd', 'ada/eur', 'ada/usd', 'ada/usd-perp', 'algo/eur', 'algo/usd', 'amp/eur', 'amp/usd', 'ape/eur', 'ape/usd', 'apt/eur', 'apt/usd', 'arb/eur', 'arb/usd', 'aster/eur', 'aster/usd']


In [5]:
# Check which of our 8 target pairs are available
TARGET_PAIRS = {
    "BTC/USD": "btcusd",
    "BTC/USDC": "btcusdc",
    "LTC/USD": "ltcusd",
    "LTC/USDC": "ltcusdc",
    "LTC/BTC": "ltcbtc",
    "SOL/USD": "solusd",
    "SOL/USDC": "solusdc",
    "SOL/BTC": "solbtc",
}

available = {}
missing = {}

for canonical, bitstamp_sym in TARGET_PAIRS.items():
    if bitstamp_sym in all_symbols:
        available[canonical] = bitstamp_sym
        print(f"  ✅ {canonical:10s} → {bitstamp_sym}")
    else:
        missing[canonical] = bitstamp_sym
        print(f"  ❌ {canonical:10s} → {bitstamp_sym} (NOT FOUND)")

print(f"\nAvailable: {len(available)}/8")
print(f"Missing:   {len(missing)}/8")
if missing:
    print(f"Missing pairs: {list(missing.keys())}")
    print("→ These will be skipped + logged at runtime (per task spec)")

  ❌ BTC/USD    → btcusd (NOT FOUND)
  ❌ BTC/USDC   → btcusdc (NOT FOUND)
  ❌ LTC/USD    → ltcusd (NOT FOUND)
  ❌ LTC/USDC   → ltcusdc (NOT FOUND)
  ❌ LTC/BTC    → ltcbtc (NOT FOUND)
  ❌ SOL/USD    → solusd (NOT FOUND)
  ❌ SOL/USDC   → solusdc (NOT FOUND)
  ❌ SOL/BTC    → solbtc (NOT FOUND)

Available: 0/8
Missing:   8/8
Missing pairs: ['BTC/USD', 'BTC/USDC', 'LTC/USD', 'LTC/USDC', 'LTC/BTC', 'SOL/USD', 'SOL/USDC', 'SOL/BTC']
→ These will be skipped + logged at runtime (per task spec)


In [6]:
# Build SymbolTranslator with ONLY available pairs
BITSTAMP_SYMBOL_MAP = available.copy()

bitstamp_translator = create_translator(
    venue="bitstamp",
    canonical_to_venue=BITSTAMP_SYMBOL_MAP,
)

# Round-trip test: canonical → venue → canonical
print("Round-trip translation test:\n")
for canonical in BITSTAMP_SYMBOL_MAP:
    venue_sym = bitstamp_translator.to_venue(canonical)
    back = bitstamp_translator.to_canonical(venue_sym)
    ok = "✅" if back == canonical else "❌"
    print(f"  {ok} {canonical} → {venue_sym} → {back}")

# USD ≠ USDC verification (DEC-001)
if "BTC/USD" in BITSTAMP_SYMBOL_MAP and "BTC/USDC" in BITSTAMP_SYMBOL_MAP:
    usd_sym = bitstamp_translator.to_venue("BTC/USD")
    usdc_sym = bitstamp_translator.to_venue("BTC/USDC")
    print(f"\nUSD ≠ USDC check: {usd_sym} ≠ {usdc_sym} → {usd_sym != usdc_sym} ✅")
else:
    print("\n⚠️  Cannot verify USD ≠ USDC — one or both pairs missing")

TypeError: create_translator() got an unexpected keyword argument 'canonical_to_venue'

## 3. Ticker Endpoint

Test `GET /api/v2/ticker/{symbol}/` to confirm it has bid/ask prices
but **NO bid/ask sizes** (per Phase A research, same issue as Gemini per LL-060).

In [ ]:
# Test ticker endpoint with BTC/USD
test_sym = BITSTAMP_SYMBOL_MAP.get("BTC/USD", "btcusd")
resp = httpx.get(f"{BASE_URL}/api/v2/ticker/{test_sym}/")
print(f"GET /api/v2/ticker/{test_sym}/")
print(f"Status: {resp.status_code}\n")

ticker = resp.json()
pprint(ticker)

# Check for bid/ask SIZE fields
print(f"\n--- Field analysis ---")
print(f"'bid' present:      {('bid' in ticker)}  → value: {ticker.get('bid')}")
print(f"'ask' present:      {('ask' in ticker)}  → value: {ticker.get('ask')}")
print(f"'bid_size' present: {('bid_size' in ticker)}")
print(f"'ask_size' present: {('ask_size' in ticker)}")
print(f"'volume' present:   {('volume' in ticker)}  → value: {ticker.get('volume')}")

has_sizes = "bid_size" in ticker or "ask_size" in ticker
if has_sizes:
    print("\n⚠️  UNEXPECTED: Ticker HAS bid/ask sizes! Reconsider using ticker vs book.")
else:
    print("\n✅ Confirmed: Ticker has bid/ask PRICES but NO SIZES → must use order book (LL-060)")

## 4. Order Book Endpoint — Primary Data Source

`GET /api/v2/order_book/{symbol}/`

This is our primary data source for TopOfBook with sizes.

**Critical format difference**: Bitstamp returns **arrays-of-arrays**, not arrays-of-objects.
- Bitstamp: `[["price", "amount"], ...]`
- Gemini: `[{"price": "...", "amount": "...", "timestamp": "..."}, ...]`

Parser must use index-based access: `entry[0]` = price, `entry[1]` = amount.

In [ ]:
# Fetch order book for BTC/USD
test_sym = BITSTAMP_SYMBOL_MAP.get("BTC/USD", "btcusd")
resp = httpx.get(f"{BASE_URL}/api/v2/order_book/{test_sym}/")
print(f"GET /api/v2/order_book/{test_sym}/")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}\n")

book = resp.json()

# Show structure
print(f"Top-level keys: {list(book.keys())}")
print(f"timestamp:      {book.get('timestamp')}")
print(f"microtimestamp: {book.get('microtimestamp')}")
print(f"bids count:     {len(book.get('bids', []))}")
print(f"asks count:     {len(book.get('asks', []))}")

print(f"\n--- Best bid (bids[0]) ---")
print(f"Type: {type(book['bids'][0])}")
print(f"Value: {book['bids'][0]}")
print(f"  [0] = price:  {book['bids'][0][0]}")
print(f"  [1] = amount: {book['bids'][0][1]}")

print(f"\n--- Best ask (asks[0]) ---")
print(f"Type: {type(book['asks'][0])}")
print(f"Value: {book['asks'][0]}")
print(f"  [0] = price:  {book['asks'][0][0]}")
print(f"  [1] = amount: {book['asks'][0][1]}")

print(f"\n--- Raw JSON (first 2 levels) ---")
print(json.dumps({
    "timestamp": book["timestamp"],
    "microtimestamp": book["microtimestamp"],
    "bids": book["bids"][:3],
    "asks": book["asks"][:3],
}, indent=2))

In [ ]:
# Fetch order books for ALL available target pairs
print("Top-of-book for all available target pairs:\n")

all_books = {}
for canonical, bitstamp_sym in BITSTAMP_SYMBOL_MAP.items():
    resp = httpx.get(f"{BASE_URL}/api/v2/order_book/{bitstamp_sym}/")
    if resp.status_code == 200:
        book = resp.json()
        all_books[canonical] = book

        if book.get("bids") and book.get("asks"):
            bid = book["bids"][0]
            ask = book["asks"][0]
            print(
                f"  {canonical:10s} → bid={bid[0]:>12s} ({bid[1]:>12s}) "
                f"| ask={ask[0]:>12s} ({ask[1]:>12s})"
            )
        else:
            print(f"  {canonical:10s} → ⚠️  Empty book: bids={len(book.get('bids', []))}, asks={len(book.get('asks', []))}")
    else:
        print(f"  {canonical:10s} → ❌ HTTP {resp.status_code}")

    time.sleep(0.2)  # Conservative delay even with 400 req/sec limit

## 5. Parse into TopOfBook

Build a prototype parser that converts Bitstamp order book responses into our `TopOfBook` dataclass.

**Key difference from Gemini parser**: index-based access for arrays-of-arrays format.
- Bitstamp: `bids[0][0]` = price, `bids[0][1]` = amount
- Gemini: `bids[0]["price"]`, `bids[0]["amount"]`

**Timestamp**: Use `microtimestamp` (microseconds) for highest precision.
- `int(microtimestamp) // 1000` → milliseconds for ts_exchange_ms

In [ ]:
def parse_bitstamp_book(
    bitstamp_symbol: str,
    canonical_pair: str,
    book_data: dict,
    ts_local_ms: int,
) -> TopOfBook:
    """
    Parse Bitstamp order book response into TopOfBook.

    Args:
        bitstamp_symbol: Bitstamp symbol (e.g., 'btcusd')
        canonical_pair: Our canonical pair (e.g., 'BTC/USD')
        book_data: The order book dict with 'bids' and 'asks' as arrays-of-arrays
        ts_local_ms: Local timestamp when data was received

    Returns:
        TopOfBook instance

    Raises:
        ValueError: If data is missing or invalid
    """
    bids = require_present(book_data.get("bids"), f"{bitstamp_symbol}.bids")
    asks = require_present(book_data.get("asks"), f"{bitstamp_symbol}.asks")

    if not bids or not asks:
        raise ValueError(f"Empty book for {bitstamp_symbol}: bids={len(bids)}, asks={len(asks)}")

    best_bid = bids[0]  # [price_str, amount_str]
    best_ask = asks[0]  # [price_str, amount_str]

    # Bitstamp provides microtimestamp (Unix microseconds as string)
    # Convert to milliseconds for ts_exchange_ms
    micro_ts = book_data.get("microtimestamp")
    if micro_ts:
        ts_exchange_ms = int(micro_ts) // 1000
    else:
        # Fallback to seconds-precision timestamp
        ts_exchange_ms = int(book_data["timestamp"]) * 1000

    return tob_from_raw(
        venue="bitstamp",
        pair=canonical_pair,
        ts_local_ms=ts_local_ms,
        ts_exchange_ms=ts_exchange_ms,
        bid_px=best_bid[0],   # Index 0 = price
        bid_sz=best_bid[1],   # Index 1 = amount
        ask_px=best_ask[0],   # Index 0 = price
        ask_sz=best_ask[1],   # Index 1 = amount
    )


print("Parser function defined ✅")

In [ ]:
# Test parser on all available pairs
ts_now = int(time.time() * 1000)
print("Parsing all available pairs into TopOfBook:\n")

for canonical, book in all_books.items():
    try:
        tob = parse_bitstamp_book(
            bitstamp_symbol=BITSTAMP_SYMBOL_MAP[canonical],
            canonical_pair=canonical,
            book_data=book,
            ts_local_ms=ts_now,
        )
        print(
            f"  ✅ {tob.pair:10s}: bid={tob.bid_px} ({tob.bid_sz}), "
            f"ask={tob.ask_px} ({tob.ask_sz}), "
            f"ts_exchange={tob.ts_exchange_ms}"
        )
    except Exception as e:
        print(f"  ❌ {canonical}: {e}")

## 6. Async httpx Pattern

Preview the production pattern: async httpx with per-pair requests.
Bitstamp has no batch endpoint for order books, so we fetch each pair individually
(same as Gemini and Coinbase).

Production connector will use `BaseAsyncConnector._fetch_tickers_per_pair()` template.

In [ ]:
async def fetch_all_books_async(
    pairs: dict[str, str],
    delay_ms: int = 150,
) -> dict[str, TopOfBook]:
    """
    Fetch all order books asynchronously with rate limiting.
    Production preview — actual connector will use BaseAsyncConnector.
    """
    results = {}
    async with httpx.AsyncClient(base_url=BASE_URL, timeout=10.0) as client:
        for canonical, bitstamp_sym in pairs.items():
            try:
                resp = await client.get(f"/api/v2/order_book/{bitstamp_sym}/")
                ts_local = int(time.time() * 1000)
                resp.raise_for_status()
                book = resp.json()
                tob = parse_bitstamp_book(bitstamp_sym, canonical, book, ts_local)
                results[canonical] = tob
            except Exception as e:
                print(f"  ⚠️  {canonical}: {e}")
            await asyncio.sleep(delay_ms / 1000)
    return results


print("Async fetch with 150ms delay (simulates production polling):\n")
start = time.time()
tobs = asyncio.run(fetch_all_books_async(BITSTAMP_SYMBOL_MAP, delay_ms=150))
elapsed = time.time() - start

print(f"\nFetched {len(tobs)} pairs in {elapsed:.2f}s")
for pair, tob in tobs.items():
    print(f"  {tob.pair:10s}: bid={tob.bid_px:>12s} ask={tob.ask_px:>12s}")

n_pairs = len(BITSTAMP_SYMBOL_MAP)
expected = n_pairs * 0.15
print(f"\n{n_pairs} pairs × 150ms = {expected:.1f}s expected, actual = {elapsed:.2f}s")
print(f"Well within 5s polling interval ✅" if elapsed < 5.0 else "⚠️ Exceeds 5s polling interval!")

## 7. Rate Limit Testing

Bitstamp docs state: 400 req/sec, 10,000 per 10 minutes.
This is extremely generous vs other exchanges (Gemini 120/min, Kraken ~1/sec).

Let's verify with burst and sustained tests.

In [ ]:
# Rapid burst test — 20 requests with no delay
test_sym = BITSTAMP_SYMBOL_MAP.get("BTC/USD", "btcusd")
print(f"Burst test: 20 rapid requests to /api/v2/order_book/{test_sym}/\n")

latencies = []
status_codes = []

for i in range(20):
    start = time.time()
    resp = httpx.get(f"{BASE_URL}/api/v2/order_book/{test_sym}/")
    elapsed_ms = (time.time() - start) * 1000
    latencies.append(elapsed_ms)
    status_codes.append(resp.status_code)
    print(f"  Request {i + 1:2d}: {resp.status_code} in {elapsed_ms:.0f}ms")

n_429 = status_codes.count(429)
avg_ms = sum(latencies) / len(latencies)
print(f"\n429 responses: {n_429}/20")
print(f"Avg latency: {avg_ms:.0f}ms")
if n_429 == 0:
    print("✅ No rate limiting hit with 20 rapid requests")
else:
    print(f"⚠️  Hit rate limit at request {status_codes.index(429) + 1}")

In [ ]:
# Sustained test — all pairs with 150ms delay (simulates production polling cycle)
print("Sustained test: all pairs with 150ms delay (simulates 1 polling cycle)\n")

sustained_latencies = []
sustained_errors = 0
start_total = time.time()

for canonical, bitstamp_sym in BITSTAMP_SYMBOL_MAP.items():
    start = time.time()
    resp = httpx.get(f"{BASE_URL}/api/v2/order_book/{bitstamp_sym}/")
    elapsed_ms = (time.time() - start) * 1000
    sustained_latencies.append(elapsed_ms)

    if resp.status_code != 200:
        sustained_errors += 1
        print(f"  {canonical}: ⚠️  {resp.status_code} in {elapsed_ms:.0f}ms")
    else:
        print(f"  {canonical}: 200 in {elapsed_ms:.0f}ms")

    time.sleep(0.15)  # 150ms delay

total_s = time.time() - start_total
avg_sustained = sum(sustained_latencies) / len(sustained_latencies)
print(f"\nTotal cycle: {total_s:.2f}s for {len(BITSTAMP_SYMBOL_MAP)} pairs")
print(f"Avg latency: {avg_sustained:.0f}ms")
print(f"Errors: {sustained_errors}")
print(f"{'✅' if sustained_errors == 0 else '⚠️'} Within 5s polling window: {total_s < 5.0}")

## 8. Error Handling

Test various error scenarios to understand Bitstamp's error response format.
Per API docs, errors include `status` or `reason` fields. Let's verify.

In [ ]:
# Test 1: Invalid symbol
print("--- Test 1: Invalid symbol ---")
resp = httpx.get(f"{BASE_URL}/api/v2/order_book/invalidpair/")
print(f"Status: {resp.status_code}")
print(f"Headers Content-Type: {resp.headers.get('content-type')}")
try:
    error_body = resp.json()
    print(f"JSON response:")
    pprint(error_body)
except Exception:
    print(f"Raw text: {resp.text[:500]}")

time.sleep(0.2)

In [ ]:
# Test 2: Wrong endpoint path
print("--- Test 2: Wrong endpoint path ---")
resp = httpx.get(f"{BASE_URL}/api/v2/nonexistent/btcusd/")
print(f"Status: {resp.status_code}")
try:
    error_body = resp.json()
    print(f"JSON response:")
    pprint(error_body)
except Exception:
    print(f"Raw text: {resp.text[:500]}")

time.sleep(0.2)

In [ ]:
# Test 3: Invalid symbol on ticker endpoint
print("--- Test 3: Invalid symbol on ticker ---")
resp = httpx.get(f"{BASE_URL}/api/v2/ticker/fakesymbol/")
print(f"Status: {resp.status_code}")
try:
    error_body = resp.json()
    print(f"JSON response:")
    pprint(error_body)
except Exception:
    print(f"Raw text: {resp.text[:500]}")

time.sleep(0.2)

In [ ]:
# Test 4: Empty symbol
print("--- Test 4: Empty symbol (no trailing path) ---")
resp = httpx.get(f"{BASE_URL}/api/v2/order_book/")
print(f"Status: {resp.status_code}")
try:
    error_body = resp.json()
    print(f"JSON response:")
    pprint(error_body)
except Exception:
    print(f"Raw text: {resp.text[:500]}")

print("\n--- Error format summary ---")
print("Document the actual error response structure here after running.")
print("Key fields to look for: 'status', 'reason', 'code', 'error', 'message'")

## 9. Currencies & Market Details

Use `/api/v2/currencies/` for precision, min sizes, withdrawal info.
Key goal: find SOL withdrawal fee (not found in web research).

In [ ]:
# Fetch all currencies
resp = httpx.get(f"{BASE_URL}/api/v2/currencies/")
print(f"GET /api/v2/currencies/")
print(f"Status: {resp.status_code}\n")

currencies_data = resp.json()
print(f"Response type: {type(currencies_data)}")

# Inspect structure
if isinstance(currencies_data, list):
    print(f"Total currencies: {len(currencies_data)}")
    if currencies_data:
        print(f"\nFirst entry structure:")
        pprint(currencies_data[0])
elif isinstance(currencies_data, dict):
    print(f"Top-level keys: {list(currencies_data.keys())}")
    for key, val in currencies_data.items():
        if isinstance(val, list) and val:
            print(f"\n{key}: {len(val)} items, first entry:")
            pprint(val[0])

In [ ]:
# Filter for our target assets: BTC, LTC, SOL, USD, USDC
TARGET_CURRENCIES = {"btc", "ltc", "sol", "usd", "usdc"}

our_currencies = {}

# Adapt based on actual response structure from cell above
if isinstance(currencies_data, list):
    for c in currencies_data:
        # Try common field names
        code = (c.get("currency") or c.get("code") or c.get("symbol") or "").lower()
        if code in TARGET_CURRENCIES:
            our_currencies[code] = c
elif isinstance(currencies_data, dict):
    for key, val in currencies_data.items():
        if isinstance(val, list):
            for c in val:
                code = (c.get("currency") or c.get("code") or c.get("symbol") or "").lower()
                if code in TARGET_CURRENCIES:
                    our_currencies[code] = c

print(f"Found {len(our_currencies)}/{len(TARGET_CURRENCIES)} target currencies\n")

for code in sorted(our_currencies):
    print(f"--- {code.upper()} ---")
    pprint(our_currencies[code])
    print()

In [ ]:
# Extract withdrawal fee info — especially SOL
print("Withdrawal fee summary:\n")
for code in ["btc", "ltc", "sol", "usd", "usdc"]:
    if code in our_currencies:
        c = our_currencies[code]
        # Look for withdrawal fee fields — adapt based on actual structure
        withdrawal_fee = c.get("withdrawal_fee") or c.get("min_withdrawal") or "(check nested fields)"
        decimals = c.get("decimals") or c.get("precision") or "?"
        print(f"  {code.upper():5s}: withdrawal_fee={withdrawal_fee}, precision={decimals}")
    else:
        print(f"  {code.upper():5s}: NOT FOUND in currencies response")

print("\n⚠️  If SOL withdrawal fee not visible here, check nested network/chain fields.")
print("   May need to inspect full currency object for SOL.")

In [ ]:
# USD vs USDC comparison (DEC-001)
print("USD vs USDC comparison (DEC-001: USD ≠ USDC):\n")
for code in ["usd", "usdc"]:
    if code in our_currencies:
        print(f"--- {code.upper()} ---")
        pprint(our_currencies[code])
        print()
    else:
        print(f"  {code.upper()}: NOT in currencies list")

## 10. Timestamp Format Deep Dive

Bitstamp provides two timestamp fields in order book responses:
- `timestamp`: Unix seconds as string (e.g., `"1643643584"`)
- `microtimestamp`: Unix microseconds as string (e.g., `"1643643584684047"`)

For ts_exchange_ms: `int(microtimestamp) // 1000`

Compare:
- Kraken: Unix seconds as float
- Coinbase: ISO 8601 with microseconds
- Gemini: Unix seconds as integer string

In [ ]:
# Analyze timestamps from fetched order books
print("Timestamp analysis across all available pairs:\n")

from datetime import datetime, timezone

for canonical, book in all_books.items():
    ts_str = book.get("timestamp", "?")
    micro_str = book.get("microtimestamp", "?")

    # Parse into human-readable
    ts_s = int(ts_str)
    micro_us = int(micro_str)
    ts_ms_from_micro = micro_us // 1000
    ts_ms_from_seconds = ts_s * 1000

    dt = datetime.fromtimestamp(ts_s, tz=timezone.utc)

    print(f"  {canonical:10s}:")
    print(f"    timestamp:      {ts_str} → {dt.isoformat()} UTC")
    print(f"    microtimestamp: {micro_str}")
    print(f"    ts_ms (micro):  {ts_ms_from_micro}")
    print(f"    ts_ms (secs):   {ts_ms_from_seconds}")
    print(f"    sub-second ms:  {ts_ms_from_micro - ts_ms_from_seconds}ms")
    print()

print("→ microtimestamp provides sub-second precision (Gemini only has second precision)")
print("→ Use int(microtimestamp) // 1000 for ts_exchange_ms")

## 11. Summary & Connector Design Notes

### Comparison Table

| Feature | Kraken | Coinbase | Gemini | Bitstamp |
|---------|--------|----------|--------|----------|
| Symbol format | XXBTZUSD | BTC-USD | btcusd | btcusd |
| BBO source | Ticker | Book | Book | Book |
| Bid/ask sizes in ticker | ✅ | N/A | ❌ | ❌ |
| Batch endpoint | ✅ | ❌ | ❌ | ❌ |
| Timestamp | Unix float | ISO 8601 | Unix int str | Unix str + microtimestamp |
| Rate limit | ~1/sec | 10/sec | 2/sec | 400/sec |
| Rate limiter interval | 500ms | 100ms | 500ms | 150ms |
| Order book format | arrays of objects | Nested pricebook | arrays of objects | **arrays of arrays** |
| SDK | python-kraken-sdk | custom httpx | custom httpx | custom httpx |

### Key Findings

1. **Pair availability**: TBD/8 available (update after running notebook)
2. **Order book format**: arrays-of-arrays — parser uses index-based access `[0]`=price, `[1]`=amount
3. **Microtimestamp**: Provides microsecond precision (better than Gemini's second precision)
4. **Rate limits**: 400 req/sec is extremely generous — 150ms interval is very conservative
5. **No batch endpoint**: Per-pair requests required (same as Gemini, Coinbase)
6. **Error format**: TBD — document after running error handling cells

### Production Connector Design

- Inherit `BaseAsyncConnector` (per DEC-018)
- Use `_fetch_tickers_per_pair()` template method
- Base URL: `https://www.bitstamp.net`
- Endpoint: `/api/v2/order_book/{symbol}/`
- Rate limiter interval: 150ms
- Parser: index-based access for arrays-of-arrays
- Timestamp: `int(microtimestamp) // 1000`
- Symbol format: lowercase, no separator (reuse `create_translator()` pattern)
- Only map pairs confirmed available (skip missing with log warning)

### Refactor Checkpoint (Coding Rule 10.9)

This is the **4th connector**. Flag any new patterns:
- Same `create_translator()` pattern across all 4 connectors → well-established, no refactor needed
- Same `_fetch_tickers_per_pair()` across Coinbase, Gemini, Bitstamp → BaseAsyncConnector handles it
- Parser structure varies per exchange (SDK, key-based, index-based) → appropriate variation, no DRY issue
- DummyRateLimiter: already flagged for extraction to tests/helpers.py (3rd caller in Gemini)

### Lessons Learned (to add to docs/LESSONS_LEARNED.md)

- **LL-062** (pending): Bitstamp order book entries are arrays-of-arrays, not arrays-of-objects.
  Rule: Never assume order book entry format. Verify response shape in notebook before building parser.
- **LL-063** (pending): Bitstamp `microtimestamp` provides microsecond precision.
  Rule: Prefer higher-precision timestamps when available. Parse as `int(str) // 1000` for ms.

In [ ]:
# Generate fixture data for tests — save raw responses
# (Run this cell after verifying all outputs above)
import json

# Pairs to capture as test fixtures
# Include a USD pair, a BTC-quoted pair, and a USDC pair (if available)
fixture_candidates = {
    "BTC/USD": "btcusd",
    "LTC/BTC": "ltcbtc",
    "BTC/USDC": "btcusdc",  # if available
}

print("Fixture responses for test data:\n")
for canonical, bitstamp_sym in fixture_candidates.items():
    if canonical not in BITSTAMP_SYMBOL_MAP:
        print(f"--- {canonical} ({bitstamp_sym}) ---")
        print(f"  ⚠️  Not available on Bitstamp — skipping fixture\n")
        continue

    resp = httpx.get(f"{BASE_URL}/api/v2/order_book/{bitstamp_sym}/")
    data = resp.json()

    # Trim to top-of-book only for fixture (keep first 2 bids/asks for testing)
    fixture = {
        "timestamp": data["timestamp"],
        "microtimestamp": data["microtimestamp"],
        "bids": data["bids"][:2],
        "asks": data["asks"][:2],
    }

    print(f"--- {canonical} ({bitstamp_sym}) ---")
    print(json.dumps(fixture, indent=2))
    print()
    time.sleep(0.2)

print("\n💡 Copy these responses to tests/fixtures/ when building the production connector.")
print("   Naming convention: bitstamp_book_btc_usd.json, bitstamp_book_ltc_btc.json, etc.")

In [ ]:
# Final summary of pair availability
print("=" * 60)
print("BITSTAMP PAIR AVAILABILITY SUMMARY")
print("=" * 60)
print(f"\nAvailable ({len(available)}/8):")
for canonical, sym in sorted(available.items()):
    print(f"  ✅ {canonical:10s} → {sym}")

if missing:
    print(f"\nMissing ({len(missing)}/8):")
    for canonical, sym in sorted(missing.items()):
        print(f"  ❌ {canonical:10s} → {sym}")
    print("\n→ Missing pairs will be excluded from BITSTAMP_SYMBOL_MAP")
    print("→ Connector will log a warning for each missing pair at startup")
    print("→ Do NOT remove these pairs from config.yaml (other exchanges may support them)")

print("\n" + "=" * 60)
print("PRODUCTION SYMBOL MAP (copy to connectors/bitstamp/symbols.py):")
print("=" * 60)
print("\nBITSTAMP_SYMBOL_MAP: dict[str, str] = {")
for canonical, sym in sorted(BITSTAMP_SYMBOL_MAP.items()):
    print(f'    "{canonical}": "{sym}",')
print("}")